In [2]:
import pandas as pd

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)

user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

print(user_movie_matrix.shape)

(6040, 3706)


In [32]:
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Type"],
    encoding="latin-1"
)

In [3]:
n_users = ratings["UserID"].nunique()
n_movies = ratings["MovieID"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

print(f"la sparsité de la matrice est de {sparsity*100:.2f} %")


la sparsité de la matrice est de 95.53 %


## Similarité entre utilisateurs

**Cosinus** : angle entre les vecteurs de notes brutes.

$$\cos(u,v) = \frac{\sum_i r_{ui}\, r_{vi}}{\sqrt{\sum_i r_{ui}^2}\;\sqrt{\sum_i r_{vi}^2}}$$

**Pearson** : même chose, mais sur les notes centrées par la moyenne de chaque utilisateur.

$$\text{pearson}(u,v) = \frac{\sum_i (r_{ui}-\bar r_u)(r_{vi}-\bar r_v)}{\sqrt{\sum_i (r_{ui}-\bar r_u)^2}\;\sqrt{\sum_i (r_{vi}-\bar r_v)^2}}$$

Les sommes portent sur les films notés par les deux utilisateurs.

**Différence** : Pearson corrige le biais de notation (utilisateur généreux vs sévère), pas le cosinus. Pearson va de -1 à 1, le cosinus de 0 à 1 avec des notes positives.

In [4]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

#On calcule la moyenne des notes des utilisateurs pour pouvoir remplacer les colonnes vides par 0, histoire qu'il n'yait pas de probleme. 
# On va centrer les notes et donc, si une note est supérieur à 0 elle est comptée positive, si inferieure a zero negative et zero si neutre
user_means = user_movie_matrix.mean(axis=1)

# 2. Centrer les notes, puis mettre 0 pour les films non notés
R_centered = user_movie_matrix.sub(user_means, axis=0).fillna(0)

# 3. Similarité cosinus entre tous les couples d'utilisateurs
sim = cosine_similarity(R_centered)
sim_df = pd.DataFrame(sim, index=user_movie_matrix.index, columns=user_movie_matrix.index)

In [5]:
## voir les voisins d'un utilisateur
## On retire l'utilisateur de la liste pour eviter d'avoir une similarité de 1 avec lui même
u = 1  # un userId
k = 10
voisins = sim_df[u].drop(u).sort_values(ascending=False).head(k)
print(voisins)

UserID
1337    0.189242
379     0.159893
5404    0.155154
49      0.148455
2607    0.148105
5739    0.146056
5635    0.144795
2495    0.139092
4159    0.138192
4724    0.135727
Name: 1, dtype: float64


In [6]:
# Pearson entre tous les couples d'utilisateurs
## corr compare les colonnes entre elles, donc on fait .T pour avoir un utilisateur par colonne
## min_periods indique le nombre de films que doivent avoir en commmun des utilisateurs pour que l'on calcule le coeff de pearson
#sim_pearson = user_movie_matrix.T.corr(method="pearson", min_periods=5)

#### Pour prédire la note d'un utilisateur pour un film on doit:
#### 1. On garde les utilisateurs qui ont noté le film i, les autres n'apportent rien
#### 2. Parmis eux on garde les k plus similaires a u
#### 3. On calcule la moyenne pondérée de leur ecarts à leur propre moyenne et on l'ajoute a la moyenne de l'utilisateur a prédire

In [7]:
film = 25
utilisateur = 25
noteurs = user_movie_matrix.index[user_movie_matrix[film].notna().drop(utilisateur)]


IndexError: boolean index did not match indexed array along axis 0; size of axis is 6040 but size of corresponding boolean axis is 6039

In [ ]:
## index sert a avoir acces a une ligne de notre matrice

In [8]:
u = ratings.index[0]                              # un utilisateur
i = ratings.columns[ratings.loc[u].isna()][0]           # un film que u n'a pas vu

raters = ratings.index[ratings[i].notna()]
print(len(raters), "utilisateurs ont noté ce film")

IndexError: index 0 is out of bounds for axis 0 with size 0

In [9]:
ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [ ]:
## .loc[u] donne la ligne de l'utilsateur u
## isna() retourne true si la note est manquante false sinon
## .columns utilise ces true/false comme filtre pour ne garder que la liste des films non notés
## [0] prend le premier élément de cette liste
u = user_movie_matrix.index[0]                              # un utilisateur
i = user_movie_matrix.columns[user_movie_matrix.loc[u].isna()][0]           # un film que u n'a pas vu


## user_movie_matrix[i] : colonne du film i
## .notna() true si la note existe, false sinon
## .index: on utilise ces true comme filtre pour garder les utilisateurs ayant noté
raters = user_movie_matrix.index[user_movie_matrix[i].notna()]
print(len(raters), "utilisateurs ont noté ce film")

701 utilisateurs ont noté ce film


In [11]:
## Mtn qu'on a la liste des utilsateurs ayant noté le film, il faut chercher ceux plus proches
sims = sim_df.loc[u, raters]
print(sims.head())

UserID
10   -0.004469
13    0.041105
18    0.029319
23    0.000193
27   -0.009751
Name: 1, dtype: float64


In [12]:
## on garde ceux avec les similarités positives et on en retient k
## nlargest trie par valeur decroissante
sims = sims[sims > 0]
k = 20
top = sims.nlargest(k)
print(top)

UserID
223     0.126104
4572    0.110868
5978    0.107933
4360    0.095656
2566    0.094968
6006    0.085618
678     0.083317
1207    0.083097
1369    0.082819
1780    0.082328
4318    0.079665
5361    0.079030
1501    0.078719
2047    0.078610
746     0.077405
4897    0.075739
119     0.074479
3155    0.074049
1956    0.073433
1087    0.072570
Name: 1, dtype: float64


## Prédiction de la note (user-based)

$$\hat r_{ui} = \bar r_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u,v)\,(r_{vi}-\bar r_v)}{\sum_{v \in N_k(u)} |\text{sim}(u,v)|}$$

| Terme | Signification |
|---|---|
| $\hat r_{ui}$ | note prédite pour l'utilisateur $u$ sur le film $i$ |
| $\bar r_u$ | note moyenne de l'utilisateur $u$ |
| $N_k(u)$ | les $k$ voisins les plus similaires à $u$ qui ont noté le film $i$ |
| $v$ | un voisin de $u$ |
| $\text{sim}(u,v)$ | similarité entre $u$ et $v$ (poids du voisin) |
| $r_{vi}$ | note donnée par le voisin $v$ au film $i$ |
| $\bar r_v$ | note moyenne du voisin $v$ |
| $r_{vi}-\bar r_v$ | écart du voisin : a-t-il aimé le film plus ou moins que d'habitude ? |
| $\sum \text{sim}(u,v)\,(r_{vi}-\bar r_v)$ | somme des écarts pondérés par la similarité |
| $\sum \lvert\text{sim}(u,v)\rvert$ | somme des poids, pour normaliser (moyenne pondérée) |

**Idée** : on part de la moyenne de $u$ et on lui ajoute l'écart moyen de ses voisins (pondéré par leur similarité) pour ce film.

In [ ]:
## il reste a calculer la note finale. cela se fait en 3 étapes
##1. On calcule l'ecart de chaque voisin a sa moyenne

deviations = user_movie_matrix.loc[top.index, i] - user_means[top.index]
print(deviations)

## la difference dit si le voisin a aimé ce film plus ou moins que d'habitude


UserID
223     1.558594
4572    0.075862
5978    0.802005
4360   -0.580645
2566   -1.363636
6006   -1.161290
678     0.322322
1207   -0.394822
1369   -1.782178
1780    0.037338
4318   -0.847917
5361   -2.620321
1501    0.273063
2047   -1.588235
746    -0.634434
4897   -1.350000
119     1.457143
3155   -1.035294
1956    0.876923
1087   -0.790816
dtype: float64


In [15]:
## on calcule la moyenne pondérée des écarts
## On multiplie l'écart de chaque voisin par sa similarité
## .sum, on additionne le numérateur
## top.abs.sum somme le denominateur. pour normaliser
weighted_dev = (top * deviations).sum() / top.abs().sum()

In [16]:
## on ajoute la moyenne de u
pred = user_means[u] + weighted_dev
print(pred)

3.8193563018616032


In [19]:
def noter(mat_user_film,moy_user,mat_similarite,k,film_non_vu,user):

    ## 1. Rechercher les utilisateurs qui ont noté le film
    noteurs = mat_user_film.index[mat_user_film[film_non_vu].notna()]

    ## 2. On cherche les utilisateurs les plus proches de notre individu parmis ceux qui ont noté le film
    sims = mat_similarite.loc[user, noteurs]

    ## on ne garde que les k utilisateurs proches ayant des similarités proches
    sims = sims[sims > 0]
    top = sims.nlargest(k)

    ## 3. On calcule maintenant le score à attribuer a notre cible en fonction des scores donnés par nos noteurs

    ## on calcule l'ecart de chaque voisin par rapport à sa moyenne 
    ecart = mat_user_film.loc[top.index, i] - moy_user[top.index]

    ## on calcule la moyenne pondérée des ecarts
    moy_pond = (top * ecart).sum() / top.abs().sum()
    ## on ajoute le chiffre a la moyenne de notre cible
    pred = moy_user[user] + moy_pond

    return pred



In [36]:
## On va ecrire une fonction pour recommander n film a notre utilisateur
def recommander(utilisateur,n,mat_user_film,moy_utilisateur,sim_df,u=20):

    """
        n represente le nombre de films qu'on souhaite recommander
        sim_df represente la matrice de similarité cosinus
        moy_utilisateur represente le vecteur des notes moyennes des utilisateurs
        u represente le nombre d'utilisateur proches qu'on souhaite utiliser pour faire notre recomandation
    
    """
    films_non_regardes = mat_user_film.colums[mat_user_film.loc[utilisateur].isna()]
    taille = len(list(films_non_regardes))
    films_test = random.sample(list(films_non_regardes,k= min(1000,taille)))

    ## on predit les scores
    film_id = []
    pred = []
    for i in films_test:
        score = noter(mat_user_film,moy_utilisateur,sim_df,u,i,utilisateur)
        film_id.append(i)
        pred.append(score)

    proposition = pd.DataFrame({"Films":film_id,"Score":pred})
    proposition = proposition.sort_values("Score",ascending=False).head(n)


    return proposition
        

In [20]:
## On va effectuer un test
## On va chercher les films qu'un utilisateur n'a pas vu et lui recommander

## prenons l'utilsateur 248
utilsateur = user_movie_matrix.index[248]
films_non_regardes = user_movie_matrix.columns[user_movie_matrix.loc[utilisateur].isna()]

print(f"les films non regardés pour notre utilisateur sont {films_non_regardes}")

les films non regardés pour notre utilisateur sont Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
       3943, 3944, 3945, 3946, 3947, 3948, 3949, 3950, 3951, 3952],
      dtype='int64', name='MovieID', length=3621)


In [23]:
## On va selectionner 10 films au hasard qu'il n'a pas regardé
import random

random.seed(42)
films_test = random.sample(list(films_non_regardes),k=1000)

In [ ]:
## on predit les scores
film_id = []
pred = []
for i in films_test:
    score = noter(user_movie_matrix,user_means,sim_df,20,i,248)
    film_id.append(i)
    pred.append(score)

resultat = {"Films":film_id,"Score":pred}
resultat = pd.DataFrame(resultat)

C:\Users\sanou\AppData\Local\Temp\ipykernel_5596\3542448446.py:19: RuntimeWarning: invalid value encountered in scalar divide
  moy_pond = (top * ecart).sum() / top.abs().sum()


In [28]:
resultat.head()

,Films,Score
0,2897,3.749816
1,478,3.162508
2,105,3.922500
3,3340,3.681797
4,1237,4.909476


In [31]:
proposition = resultat.sort_values("Score",ascending=False).head(20)
proposition.head()

,Films,Score
932,3603,5.609108
49,439,5.451105
413,1420,5.404063
557,2905,5.325291
362,3517,5.287631


In [34]:
final = pd.merge(proposition,movies[["MovieID","Title"]],left_on="Films",right_on="MovieID")
final

,Films,Score,MovieID,Title
0,3603,5.609108,3603,"Gay Deceivers, The (1969)"
1,439,5.451105,439,Dangerous Game (1993)
2,1420,5.404063,1420,Message to Love: The Isle of Wight Festival (1...
3,2905,5.325291,2905,Sanjuro (1962)
4,3517,5.287631,3517,"Bells, The (1926)"
5,657,5.278610,657,Yankee Zulu (1994)
6,2931,5.276229,2931,Time of the Gypsies (Dom za vesanje) (1989)
7,3091,5.213853,3091,Kagemusha (1980)
8,2762,5.162365,2762,"Sixth Sense, The (1999)"
9,745,5.153625,745,"Close Shave, A (1995)"
